# Nested Booking Limits — Interactive Dashboard
Run either collapsed dashboard cell using its play button; Python code is hidden by default.


In [ ]:
#@title Nested Booking Limits — Interactive Dashboard { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

FARES = [('Promo',79), ('Saver',120), ('Standard',190), ('Flex',320)]
CAPACITY = 180
INITIAL_SOLD = [20,20,20,20]
INITIAL_LIMITS = [60,100,140,180]
# Identical illustrative future requests for both policies: low fares arrive first.
REQUESTS = [55,35,30,40]
sold = INITIAL_SOLD.copy()

css = widgets.HTML("""<style>
.rm{max-width:1050px;font-family:Arial,sans-serif;color:#14364a}
.rm h2{margin:5px 0}.rm h3{margin:16px 0 8px}
.rm small,.rm-note{color:#64748b;font-size:12px}
.rm-stats{display:flex;gap:10px;flex-wrap:wrap;margin:14px 0}
.rm-stat{flex:1;min-width:140px;background:#eef3f8;border-radius:10px;padding:13px}
.rm-stat b{display:block;font-size:25px;margin:3px 0}
.rm table{border-collapse:collapse;width:100%;font-size:14px;margin:12px 0}
.rm th{background:#e8f1f7;text-align:left;padding:9px}
.rm td{padding:9px;border-bottom:1px solid #dbe2e8}
.rm-good{color:#087e4e;font-weight:bold}.rm-bad{color:#b91c1c;font-weight:bold}
.rm-track{height:12px;background:#e5e7eb;border-radius:8px;overflow:hidden}
.rm-fill{height:100%;background:#08739a}
</style>""")
output = widgets.Output()
controls = widgets.Output()
buttons = []
sliders = [
    widgets.IntSlider(value=v,min=0,max=180,step=5,description=label,
        style={'description_width':'125px'},layout=widgets.Layout(width='430px'),
        continuous_update=False)
    for v,label in [(60,'Promo limit'),(100,'Saver limit'),(140,'Standard limit')]
]

def limits():
    return [w.value for w in sliders]+[CAPACITY]

def revenue(counts):
    return sum(n*fare for n,(_,fare) in zip(counts,FARES))

def available(counts, caps):
    left=CAPACITY-sum(counts)
    return [max(0,min(left,caps[i]-sum(counts[:i+1]))) for i in range(4)]

def forecast(nested):
    """Apply protection to future requests, never undo existing sales."""
    counts=sold.copy()
    accepted=[0]*4
    rejected=[0]*4
    caps=limits() if nested else [CAPACITY]*4
    for i,n in enumerate(REQUESTS):
        for _ in range(n):
            if sum(counts)<CAPACITY and (not nested or sum(counts[:i+1])<caps[i]):
                counts[i]+=1
                accepted[i]+=1
            else:
                rejected[i]+=1
    return counts,accepted,rejected

def render():
    caps=limits()
    avail=available(sold,caps)
    total=sum(sold)
    rows=''.join(
        f'<tr><td><b>{name}</b><br><small>USD {fare}</small></td>'
        f'<td>{sold[i]}</td><td>{caps[i]}</td><td>{avail[i]}</td>'
        f'<td>USD {sold[i]*fare:,}</td>'
        f'<td class="{"rm-good" if avail[i] else "rm-bad"}">'
        f'{"OPEN" if avail[i] else "CLOSED"}</td></tr>'
        for i,(name,fare) in enumerate(FARES)
    )
    protected,pa,pr=forecast(True)
    baseline,ba,br=forecast(False)
    delta=revenue(protected)-revenue(baseline)
    compare_rows=''.join(
        f'<tr><td>{name}</td><td>{pa[i]}</td><td>{ba[i]}</td>'
        f'<td>{pr[i]}</td><td>{br[i]}</td></tr>'
        for i,(name,_) in enumerate(FARES)
    )
    sign='+' if delta>0 else '-' if delta<0 else ''
    html=f"""
    <div class="rm">
      <small>ILLUSTRATIVE REVENUE MANAGEMENT DASHBOARD</small>
      <h2>Nested Booking Limits | Flight Inventory Control</h2>
      <small>180 seats · Economy cabin · D−14 (14 days before departure) · Illustrative fares and demand</small>
      <div class="rm-stats">
        <div class="rm-stat"><small>Seats sold</small><b>{total}</b></div>
        <div class="rm-stat"><small>Remaining seats</small><b>{CAPACITY-total}</b></div>
        <div class="rm-stat"><small>Load factor</small><b>{total/CAPACITY:.1%}</b></div>
        <div class="rm-stat"><small>Ticket revenue to date</small><b>USD {revenue(sold):,}</b></div>
      </div>
      <div class="rm-track"><div class="rm-fill" style="width:{100*total/CAPACITY:.2f}%"></div></div>
      <table><thead><tr><th>Fare class</th><th>Sold</th><th>Nested limit</th>
      <th>Available</th><th>Revenue to date</th><th>Status</th></tr></thead>
      <tbody>{rows}</tbody></table>
      <div class="rm-note">Available counts overlap and cannot be added. Ticket revenue excludes ancillary sales and costs.</div>
      <h3>Revenue comparison: identical future booking requests</h3>
      <small>From D−14 through departure, future requests arrive in order: Promo {REQUESTS[0]},
        Saver {REQUESTS[1]}, Standard {REQUESTS[2]}, Flex {REQUESTS[3]}.
        Only the inventory policy changes.</small>
      <div class="rm-stats">
        <div class="rm-stat"><small>WITH nested protection · final ticket revenue</small>
          <b>USD {revenue(protected):,}</b><small>{sum(protected)} total seats sold</small></div>
        <div class="rm-stat"><small>WITHOUT protection · first come, first served</small>
          <b>USD {revenue(baseline):,}</b><small>{sum(baseline)} total seats sold</small></div>
      </div>
      <div class="{"rm-good" if delta>=0 else "rm-bad"}">
      Revenue difference (protection minus no protection): {sign}USD {abs(delta):,}</div>
      <table><thead><tr><th>Future fare requests</th>
        <th>Accepted with protection</th><th>Accepted without</th>
        <th>Rejected with protection</th><th>Rejected without</th>
        </tr></thead><tbody>{compare_rows}</tbody></table>
      <div class="rm-note">Deterministic classroom scenario, not an optimized or
      statistically expected-revenue forecast. Protection can help or hurt when
      demand changes. Protection applies to future bookings; past sales remain.</div>
    </div>"""
    with output:
        clear_output(wait=True)
        display(HTML(html))
    for i,button in enumerate(buttons):
        button.disabled=avail[i]<=0

def slider_changed(change=None):
    # Cumulative limits must remain nondecreasing.
    a,b,c=sliders
    a.max=b.value
    b.min=a.value
    b.max=c.value
    c.min=b.value
    render()

for slider in sliders:
    slider.observe(slider_changed,names='value')

def book(i):
    if available(sold,limits())[i]>0:
        sold[i]+=1
        render()

for i,(name,fare) in enumerate(FARES):
    button=widgets.Button(description=f'+1 {name} (USD {fare})',
                          layout=widgets.Layout(width='205px'))
    button.on_click(lambda _,i=i:book(i))
    buttons.append(button)

reset=widgets.Button(description='↻ Reset example',button_style='info',
                     layout=widgets.Layout(width='185px'))
def reset_all(_):
    sold[:]=INITIAL_SOLD
    sliders[0].value=0
    sliders[1].value=0
    sliders[2].value=INITIAL_LIMITS[2]
    sliders[1].value=INITIAL_LIMITS[1]
    sliders[0].value=INITIAL_LIMITS[0]
    render()
reset.on_click(reset_all)

display(css,output,
    widgets.HTML('<b>Accept one booking:</b>'),
    widgets.HBox(buttons,layout=widgets.Layout(flex_flow='row wrap',gap='8px')),
    widgets.HTML('<b>Adjust nested booking limits (cumulative):</b>'),
    widgets.VBox(sliders),reset)
slider_changed()


## Activity 2 — Scaffolded Seat-Protection Decision

**Scenario:** A 180-seat flight has **160 seats booked**, leaving 20. Today, **15 Saver customers** request tickets at **$120**. Later, Flex customers may request tickets at **$320**.

### Stage 1 — Fixed decision

Use the fixed forecast below to compare two decisions.

| Later Flex requests | 0 | 5 | 10 | 15 | 20 |
|---|---:|---:|---:|---:|---:|
| Probability | 15% | 20% | 30% | 25% | 10% |

- **A:** Accept all 15 Saver customers now (5 seats remain for Flex).
- **B:** Accept 5 Saver customers and protect 15 seats for Flex.

**Question 1:** Estimate revenue today, expected later revenue and expected total revenue for A and B. Which decision is better under the fixed forecast? Run the notebook to check your calculations.

### Stage 2 — Introduce changing demand probabilities

After completing Stage 1, click **Continue to Stage 2** in the notebook.

**Question 2:** Adjust both the number of seats protected and the probability that **any Flex demand arrives**. Compare the expected-revenue curves at 40%, 85% (baseline) and 95%.

**Question 3:** How does the revenue-maximizing protection level change as the chance of Flex demand increases? Explain the airline's trade-off.

*Conditional on positive Flex demand, the relative distribution of 5, 10, 15 and 20 requests remains fixed. Existing ticket revenue is excluded from both stages.*

In [ ]:
#@title Activity 2 — Fixed Decision, Then Probability Sensitivity { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt

SEATS_LEFT, SAVER_REQUESTS = 20, 15
SAVER_FARE, FLEX_FARE = 120, 320
DEMAND_LEVELS = [0, 5, 10, 15, 20]
FIXED_PROBS = [0.15, 0.20, 0.30, 0.25, 0.10]
BASE_PROB_ANY = 0.85
POSITIVE_WEIGHTS = [p/BASE_PROB_ANY for p in FIXED_PROBS[1:]]

def probs_for(prob_any):
    return [1-prob_any] + [prob_any*q for q in POSITIVE_WEIGHTS]

def revenue_for(protected, prob_any=BASE_PROB_ANY):
    saver = min(SAVER_REQUESTS, SEATS_LEFT-protected)
    rev_now = saver*SAVER_FARE
    flex_seats = SEATS_LEFT-saver
    rev_later = sum(prob*min(demand,flex_seats)*FLEX_FARE
                    for demand,prob in zip(DEMAND_LEVELS,probs_for(prob_any)))
    return saver,rev_now,rev_later,rev_now+rev_later

# STAGE 1 — fixed two-policy comparison, with no adjustable controls.
a = revenue_for(5)
b = revenue_for(15)
fixed_html = f"""
<div style="font-family:Arial;color:#103952;max-width:900px">
<h3>Stage 1 · Fixed demand forecast</h3>
<p>180-seat flight · 160 sold · Saver USD 120 · Flex USD 320</p>
<table style="width:100%;border-collapse:collapse;text-align:right">
<tr style="background:#e8f1f7"><th style="text-align:left">Policy</th><th>Saver accepted</th>
<th>Revenue now</th><th>Expected later Flex revenue</th><th>Expected total</th></tr>
<tr><td style="text-align:left">A · Accept all 15</td><td>{a[0]}</td>
<td>USD {a[1]:,.0f}</td><td>USD {a[2]:,.0f}</td><td><b>USD {a[3]:,.0f}</b></td></tr>
<tr><td style="text-align:left">B · Protect 15</td><td>{b[0]}</td>
<td>USD {b[1]:,.0f}</td><td>USD {b[2]:,.0f}</td><td><b>USD {b[3]:,.0f}</b></td></tr>
</table><p style="font-size:12px;color:#64748b">Incremental ticket revenue only;
existing 160 bookings excluded.</p></div>
"""
display(HTML(fixed_html))

# The fixed revenue graph is visible in Stage 1. Stage 2 unlocks its controls.
continue_btn = widgets.Button(description='Continue to Stage 2 →',
    button_style='info',layout=widgets.Layout(width='235px'))
seats = widgets.IntSlider(value=15,min=5,max=20,step=1,
    description='Seats protected',continuous_update=False,
    style={'description_width':'190px'},layout=widgets.Layout(width='525px'))
chance = widgets.IntSlider(value=85,min=0,max=100,step=5,
    description='Chance of any Flex demand',continuous_update=False,
    style={'description_width':'190px'},layout=widgets.Layout(width='525px'))
reset_btn = widgets.Button(description='Reset stage 2',layout=widgets.Layout(width='175px'))
stage2_output = widgets.Output()
stage2 = widgets.VBox([
    widgets.HTML('<b>Stage 2 · Vary demand likelihood and protection level</b>'),
    seats,chance,reset_btn])
stage2.layout.display='none'

def redraw(change=None):
    n = seats.value
    prob_any = chance.value/100
    saver,now,later,total = revenue_for(n,prob_any)
    x=list(range(5,21))
    y=[revenue_for(v,prob_any)[3] for v in x]
    with stage2_output:
        clear_output(wait=True)
        display(HTML(
            '<div style="font-family:Arial;color:#103952">'
            f'<p>Protect <b>{n}</b> seats; accept <b>{saver}</b> Saver requests. '
            f'Revenue today: USD {now:,.0f}; expected later: USD {later:,.0f}; '
            f'<b>expected total: USD {total:,.0f}</b>.</p></div>'))
        fig,ax=plt.subplots(figsize=(9,3.5))
        ax.plot(x,y,marker='o',markersize=4,linewidth=2,label='Expected revenue')
        ax.scatter([n],[total],color='#e77628',s=95,zorder=3,label='Selected decision')
        ax.set(xlabel='Seats protected for later Flex bookings',
            ylabel='Expected incremental revenue (USD)',
            title=f'Expected revenue | Chance of some Flex demand: {chance.value}%')
        ax.set_xticks(x)
        ax.grid(alpha=0.25)
        ax.legend(loc='lower left')
        fig.tight_layout()
        display(fig)
        plt.close(fig)

def reveal(_):
    stage2.layout.display=''
    continue_btn.layout.display='none'
    redraw()

def reset_stage2(_):
    seats.value=15
    chance.value=85
    redraw()

seats.observe(redraw,names='value')
chance.observe(redraw,names='value')
continue_btn.on_click(reveal)
reset_btn.on_click(reset_stage2)
display(stage2_output,continue_btn,stage2)
redraw()
